# Entrenamiento

Este notebook entrena un modelo YOLO para detectar y contar **glóbulos rojos (GR)**, **glóbulos blancos (GB)** y **plaquetas (PQT)**.

### Antes de correrlo, importante:

- Este notebook recibe **cualquier dataset exportado en formato YOLOv8 desde Roboflow** (un zip con carpetas `train/`, `valid/`, `test/` y un archivo `data.yaml` adentro). No le importa si ese dataset es uno solo o si viene de combinar varias versiones (como el `dataset_final.zip` que genera el Merge).

- **Activar la GPU antes de correr nada:** arriba, `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`. Sin esto el entrenamiento puede tardar muchas horas en vez de minutos.

- **Revisá las clases de tu dataset antes de entrenar.** Las clases que usamos en este proyecto son:
  - `GB` = glóbulo blanco
  - `GR` = glóbulo rojo
  - `PQT` = plaqueta

  El **orden** en que aparecen en el `data.yaml` define qué número le asigna YOLO a cada clase internamente (por ejemplo `GB`=0, `GR`=1, `PQT`=2). Si tu dataset tiene las clases en otro orden o con otros nombres, FIJATE bien antes de entrenar - si el orden no coincide con cómo fueron etiquetadas las imágenes, el modelo va a aprender las clases cruzadas (por ejemplo, va a pensar que un glóbulo rojo es una plaqueta) y los resultados van a ser un desastre sin que se note el error a simple vista.

  Este notebook imprime el contenido del `data.yaml` automáticamente (Celda 3) para que lo revises antes de seguir.

## Celda 1 - Instalar dependencias

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import os
import zipfile
import shutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Celda 2 - Subir el dataset (zip exportado de Roboflow, o `dataset_final.zip` del Merge)

In [ ]:
from google.colab import files

print('Subí el zip del dataset (formato YOLOv8)')
uploaded_dataset = files.upload()
nombre_zip_dataset = list(uploaded_dataset.keys())[0]

Subí el zip del dataset (formato YOLOv8)


Saving dataset_final.zip to dataset_final.zip


## Celda 3 - Descomprimir y encontrar el `data.yaml`

**Revisá el contenido que se imprime al final.** Confirmá que `nc` (número de clases) y `names` (nombres y orden) coinciden con lo que esperás: `['GB', 'GR', 'PQT']`.

In [ ]:
carpeta_dataset = '/content/dataset'

with zipfile.ZipFile(nombre_zip_dataset, 'r') as zip_ref:
    zip_ref.extractall(carpeta_dataset)

# Buscamos el data.yaml automáticamente, sin importar en qué subcarpeta haya quedado
ruta_yaml = None
for root, dirs, files_ in os.walk(carpeta_dataset):
    if 'data.yaml' in files_:
        ruta_yaml = os.path.join(root, 'data.yaml')
        break

if ruta_yaml is None:
    print('No se encontró data.yaml dentro del zip. Revisá que el zip tenga la estructura correcta de YOLO.')
else:
    print(f'data.yaml encontrado en: {ruta_yaml}')
    print('\n--- CONTENIDO DEL data.yaml (revisá nc y names antes de seguir) ---\n')
    with open(ruta_yaml, 'r') as f:
        print(f.read())

data.yaml encontrado en: /content/dataset/data.yaml

--- CONTENIDO DEL data.yaml (revisá nc y names antes de seguir) ---

train: /content/dataset_combinado/train/images
val: /content/dataset_combinado/valid/images
test: /content/dataset_combinado/test/images

nc: 3
names: ['GB', 'GR', 'PQT']



In [ ]:
# Corregimos las rutas del data.yaml para que apunten a la carpeta real de esta sesión
carpeta_yaml = os.path.dirname(ruta_yaml)

nuevo_contenido = f'''train: {carpeta_yaml}/train/images
val: {carpeta_yaml}/valid/images
test: {carpeta_yaml}/test/images

nc: 3
names: ['GB', 'GR', 'PQT']
'''

with open(ruta_yaml, 'w') as f:
    f.write(nuevo_contenido)

print('data.yaml corregido:')
print(nuevo_contenido)

data.yaml corregido:
train: /content/dataset/train/images
val: /content/dataset/valid/images
test: /content/dataset/test/images

nc: 3
names: ['GB', 'GR', 'PQT']



## Celda 4 - Chequear que la GPU esté activa

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
print('Nombre de la GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Sin GPU - activala en Entorno de ejecución → Cambiar tipo de entorno de ejecución')

GPU disponible: True
Nombre de la GPU: Tesla T4


## Celda 5 - Cargar el modelo base de YOLO

`yolov8n.pt` es la versión "nano" - la más liviana y rápida, ideal para empezar y para datasets chicos como el nuestro. Ya viene preentrenada con millones de imágenes, así que no arranca de cero.

In [ ]:
MODELO = 'yolov8n.pt'
model = YOLO(MODELO)

## Celda 6 - Entrenar el modelo

Esta celda es la que tarda (puede ser desde 30 minutos hasta un par de horas, según el tamaño del dataset y si hay GPU activa).

- `epochs=100` - cantidad máxima de vueltas que el modelo da sobre todo el dataset
- `patience=20` - si no mejora en 20 épocas seguidas, el entrenamiento se corta solo (no hace falta esperar las 100 si ya no está mejorando)
- `imgsz=640` - tamaño al que se redimensionan las imágenes, el estándar de YOLO
- `name=` - el nombre con el que se va a guardar esta corrida. **Cambialo si vas a entrenar varias veces**, para no pisar resultados anteriores (por ejemplo `glob_y_plaquetas_v2`, `glob_y_plaquetas_v3`, etc.)

In [ ]:
results = model.train(
    data=ruta_yaml,
    epochs=100,
    imgsz=640,
    batch=8,
    patience=20,
    device=0,
    workers=2,
    project='hematologia',
    name='globulos_y_plaquetas',
    save=True
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=globulos_y_plaquetas-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

## Celda 7 - Ver las métricas del modelo entrenado

Guía rápida para leer los resultados:
- **Precision:** de todo lo que el modelo dice que es una clase (ej. plaqueta), qué porcentaje realmente lo es. Precision baja = el modelo marca cosas de más (falsos positivos).
- **Recall:** de todos los objetos reales de una clase, qué porcentaje el modelo logra encontrar. Recall bajo = al modelo se le escapan objetos reales (falsos negativos).
- **mAP50:** una medida combinada de qué tan bien detecta y ubica los objetos, considerando que la detección sea correcta con al menos 50% de superposición con el objeto real. Cuanto más cerca de 1, mejor.
- **mAP50-95:** lo mismo pero exigiendo mayor precisión en la ubicación del recuadro. Es más estricto que el anterior.

In [ ]:
metrics = model.val()

print('\n===== RESULTADOS (sobre el set de validación) =====')
print(f'mAP50      : {metrics.box.map50:.4f}')
print(f'mAP50-95   : {metrics.box.map:.4f}')
print(f'Precision  : {metrics.box.mp:.4f}')
print(f'Recall     : {metrics.box.mr:.4f}')

## Celda 8 (opcional) - Ver las métricas sobre el set de TEST

El set de validación (celda anterior) se usó indirectamente durante el entrenamiento para elegir el mejor modelo. El set de **test** el modelo nunca lo vio, así que da una medida más honesta de cómo se va a comportar con imágenes totalmente nuevas.

In [ ]:
metrics_test = model.val(split='test')

print('\n===== RESULTADOS (sobre el set de TEST) =====')
print(f'mAP50      : {metrics_test.box.map50:.4f}')
print(f'mAP50-95   : {metrics_test.box.map:.4f}')
print(f'Precision  : {metrics_test.box.mp:.4f}')
print(f'Recall     : {metrics_test.box.mr:.4f}')

## Celda 9 - Guardar todo (resultados, gráficas y el modelo entrenado) en un zip

Esto comprime la carpeta `runs` completa (incluye `best.pt`, que es el modelo final entrenado). Ese zip es el que después se usa en el notebook de **Detección** para analizar imágenes nuevas sin tener que volver a entrenar.

In [ ]:
!zip -rq runs.zip /content/runs
print('runs.zip generado correctamente')

## Celda 10 - Descargar runs.zip a la computadora

In [ ]:
from google.colab import files
files.download('runs.zip')